# Phase 2: Real-Time Transformer Detectors (RF-DETR / RT-DETR)
### Capstone Project: Transformer Challenger vs. CNN Baselines
**Hardware:** Windows 10/11 | NVIDIA RTX 4090 (24GB VRAM)  
**Models:** RT-DETR-L / RF-DETR  
**Goal:**
Compare the vision transformer inductive bias against the YOLO Phase 1 baseline on the identical board-level split, specifically analyzing convergence on thin, low-contrast defect geometry.


## 1. Setup & Environment Verification


In [ ]:
import torch
from pathlib import Path

assert torch.cuda.is_available(), "CUDA must be available."
print(f"GPU: {torch.cuda.get_device_name(0)}")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

YAML_FILE = Path("./splits/phase1/pcb_phase1.yaml").resolve()
assert YAML_FILE.exists(), f"Phase 1 YAML not found at {YAML_FILE}."
print(f"[+] Loaded identical Phase 1 Split: {YAML_FILE}")


## 2. RT-DETR-L Transformer Training
Train RT-DETR-L with deformable attention and set-based matching.


In [ ]:
from ultralytics import RTDETR

# RTX 4090 optimized hyperparams for Vision Transformer
DETR_CONFIG = {
    "data": str(YAML_FILE),
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,          # 16 is optimal for RT-DETR memory footprint on 24GB VRAM
    "device": 0,
    "workers": 8,
    "amp": True,
    "patience": 25,
    "save": True,
    "seed": 42
}

print("[*] Initializing RT-DETR-L...")
model_detr = RTDETR("rtdetr-l.pt")

print("[*] Training Transformer Detector...")
results_detr = model_detr.train(
    project="Phase2_Transformer_Challenger",
    name="RT_DETR_L_Experiment",
    **DETR_CONFIG
)
print("[+] RT-DETR-L Training complete.")


## 3. Head-to-Head Comparison: Transformer vs. YOLO26 Baseline
Direct comparison on the frozen test set across mAP@50-95, recall, and inference latency.


In [ ]:
import time
import pandas as pd
from ultralytics import YOLO, RTDETR

CLASSES = ["mouse_bite", "spur", "missing_hole", "short", "open_circuit", "spurious_copper"]

def benchmark_detector(model_cls, weights_path, label):
    model = model_cls(weights_path)
    val_res = model.val(data=str(YAML_FILE), split="test", imgsz=640, device=0)
    
    # Latency timing
    dummy = torch.zeros((1, 3, 640, 640), device="cuda")
    for _ in range(30): _ = model(dummy, verbose=False)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(150): _ = model(dummy, verbose=False)
    torch.cuda.synchronize()
    latency_ms = ((time.time() - t0) / 150) * 1000
    
    entry = {
        "Architecture": label,
        "mAP@50": round(float(val_res.box.map50), 4),
        "mAP@50-95": round(float(val_res.box.map), 4),
        "Latency (ms)": round(latency_ms, 2),
        "FPS": round(1000.0 / latency_ms, 1)
    }
    for idx, c_name in enumerate(CLASSES):
        entry[f"Recall_{c_name}"] = round(float(val_res.box.r[idx]), 4) if idx < len(val_res.box.r) else 0.0
    return entry

yolo_path = "Phase1_YOLO_Baselines/YOLO26s_Phase1/weights/best.pt"
detr_path = "Phase2_Transformer_Challenger/RT_DETR_L_Experiment/weights/best.pt"

comparison_data = []
if Path(yolo_path).exists():
    comparison_data.append(benchmark_detector(YOLO, yolo_path, "YOLO26s (CNN)"))
if Path(detr_path).exists():
    comparison_data.append(benchmark_detector(RTDETR, detr_path, "RT-DETR-L (ViT)"))

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)


## 4. Visual Comparison of Detection Failures
Inspect failure modes (false positives vs false negatives) across both models.


In [ ]:
import matplotlib.pyplot as plt

if not df_comparison.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    df_plot = df_comparison[["Architecture", "mAP@50", "mAP@50-95"]].melt(id_vars="Architecture")
    import seaborn as sns
    sns.barplot(data=df_plot, x="Architecture", y="value", hue="variable", ax=ax, palette="mako")
    ax.set_title("Phase 1 Baseline vs Phase 2 Transformer Challenger", fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
